## Causal attention mechanism

In [1]:
import numpy as np
import torch
import math

In [2]:
# input embeddings
inputs = torch.tensor([
   [0.43, 0.15, 0.89], # Your 
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]  # step
])

In [3]:
x_2 = inputs[1]
x_2

tensor([0.5500, 0.8700, 0.6600])

In [ ]:
dimension_input = inputs.shape[1]
dimension_output = 2

In [9]:
# set the seed value to get same output every time
torch.manual_seed(123)

# create the weight metrices for Q,K,V
weight_query = torch.nn.Parameter(torch.rand(dimension_input, dimension_output), requires_grad=False)
weight_key = torch.nn.Parameter(torch.rand(dimension_input, dimension_output), requires_grad=False)
weight_value = torch.nn.Parameter(torch.rand(dimension_input, dimension_output), requires_grad=False)

## compute Q,K,V

In [10]:
# compute Q
queries = inputs @ weight_query

In [11]:
keys = inputs @ weight_key

In [12]:
values = inputs @ weight_value

In [13]:
# find the attention scores for all the input embeddings
attention_scores = queries @ keys.T
attention_scores

tensor([[0.9231, 1.3545, 1.3241, 0.7910, 0.4032, 1.1330],
        [1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
        [1.2544, 1.8284, 1.7877, 1.0654, 0.5508, 1.5238],
        [0.6973, 1.0167, 0.9941, 0.5925, 0.3061, 0.8475],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707, 0.7307],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])

In [14]:
# get the dimension of keys 
dimension_keys = keys.shape[-1]

# calculate scaled weights
scaled_weights = attention_scores / math.sqrt(dimension_keys)

In [16]:
# calculate attention weights
attention_weights = torch.softmax(scaled_weights, dim=-1)
attention_weights

tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])

## create the casual mask

In [19]:
# define context length
context_length = inputs.shape[0]
context_length
# craete the ones tensor
tensor_ones = torch.ones(context_length, context_length)
tensor_ones

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [22]:
# create the upper triangular matrix
upper_triangular_mtrx = torch.triu(tensor_ones, diagonal=1)
upper_triangular_mtrx

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

## replace the 1s with -inf

In [23]:
causal_mask = upper_triangular_mtrx.bool()
causal_mask

tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])

In [24]:
causal_attention_scores = attention_scores.masked_fill(causal_mask, -torch.inf)
causal_attention_scores

tensor([[0.9231,   -inf,   -inf,   -inf,   -inf,   -inf],
        [1.2705, 1.8524,   -inf,   -inf,   -inf,   -inf],
        [1.2544, 1.8284, 1.7877,   -inf,   -inf,   -inf],
        [0.6973, 1.0167, 0.9941, 0.5925,   -inf,   -inf],
        [0.6114, 0.8819, 0.8626, 0.5121, 0.2707,   -inf],
        [0.8995, 1.3165, 1.2871, 0.7682, 0.3937, 1.0996]])

In [ ]:
causal_scaled_weights = causal_attention_scores / math.sqrt(dimension_keys)
causal_scaled_weights

tensor([[0.6528,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.8984, 1.3098,   -inf,   -inf,   -inf,   -inf],
        [0.8870, 1.2929, 1.2641,   -inf,   -inf,   -inf],
        [0.4930, 0.7189, 0.7029, 0.4190,   -inf,   -inf],
        [0.4323, 0.6236, 0.6099, 0.3621, 0.1914,   -inf],
        [0.6361, 0.9309, 0.9101, 0.5432, 0.2784, 0.7776]])

In [27]:
causal_attention_weights = torch.softmax(causal_scaled_weights, dim=-1)
causal_attention_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3986, 0.6014, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2526, 0.3791, 0.3683, 0.0000, 0.0000, 0.0000],
        [0.2265, 0.2839, 0.2794, 0.2103, 0.0000, 0.0000],
        [0.1952, 0.2363, 0.2331, 0.1820, 0.1534, 0.0000],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])

### calculate the context vector 

In [28]:
context_vector = causal_attention_weights @ values
context_vector

tensor([[0.1855, 0.8812],
        [0.3116, 0.9549],
        [0.3395, 0.9652],
        [0.3129, 0.8747],
        [0.2865, 0.7897],
        [0.2990, 0.8040]])